# FIP 수집 (2021–2025)

유사 투수 매칭(파이프라인 5번)의 "더 낮은 FIP" 조건에 쓸 시즌별 FIP를 만든다.

- 출처: MLB Stats API (공식, 무료). 선수 ID가 Statcast `pitcher` 컬럼과 같은 **MLBID**라 클러스터 json과 ID로 바로 합칠 수 있다.
- FIP = (13×HR + 3×(BB+HBP) − 2×K) / IP + 시즌 상수
- 시즌 상수는 그 시즌 리그 전체 ERA와 FIP가 같아지도록 같은 데이터에서 계산한다. FanGraphs 값과 소수 둘째 자리 정도 차이가 날 수 있다.
- 결과: `data/processed/fip_2021_2025.csv` (player_id, player_name_fip, game_year, IP, FIP 등)

## Step 1. 시즌별 투수 기록 받기

In [1]:
import requests
import pandas as pd

def fetch_pitching(season):
    url = "https://statsapi.mlb.com/api/v1/stats"
    params = {"stats": "season", "group": "pitching", "season": season,
              "sportId": 1, "playerPool": "All", "limit": 5000}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    rows = []
    for s in r.json()["stats"][0]["splits"]:
        st = s["stat"]
        whole, frac = st.get("inningsPitched", "0.0").split(".")
        rows.append({
            "player_id": s["player"]["id"],
            "player_name_fip": s["player"]["fullName"],
            "game_year": season,
            "outs": int(whole) * 3 + int(frac),   # 145.1이닝 = 436아웃
            "hr": st.get("homeRuns", 0),
            "bb": st.get("baseOnBalls", 0),
            "hbp": st.get("hitByPitch", 0),
            "so": st.get("strikeOuts", 0),
            "er": st.get("earnedRuns", 0),
        })
    return pd.DataFrame(rows)

raw = pd.concat([fetch_pitching(y) for y in range(2021, 2026)], ignore_index=True)
print(raw.shape)

(4371, 9)


## Step 2. FIP 계산

시즌 중 이적한 투수가 여러 줄로 나오면 합친 뒤, 시즌별 리그 상수를 구해 FIP를 계산한다.

In [2]:
count_cols = ["outs", "hr", "bb", "hbp", "so", "er"]
stats = raw.groupby(["player_id", "game_year"], as_index=False).agg(
    {"player_name_fip": "first", **{c: "sum" for c in count_cols}})
stats["IP"] = stats["outs"] / 3

lg = stats.groupby("game_year")[count_cols].sum()
lg_ip = lg["outs"] / 3
lg_era = 9 * lg["er"] / lg_ip
cfip = lg_era - (13 * lg["hr"] + 3 * (lg["bb"] + lg["hbp"]) - 2 * lg["so"]) / lg_ip
print("시즌별 FIP 상수 (3점대 초반이면 정상):")
print(cfip.round(3))

stats = stats[stats["IP"] > 0].copy()
stats["FIP"] = ((13 * stats["hr"] + 3 * (stats["bb"] + stats["hbp"]) - 2 * stats["so"]) / stats["IP"]
                + stats["game_year"].map(cfip))
print(stats.sort_values("IP", ascending=False).head(10)[["player_name_fip", "game_year", "IP", "FIP"]])

시즌별 FIP 상수 (3점대 초반이면 정상):
game_year
2021    3.170
2022    3.112
2023    3.255
2024    3.166
2025    3.135
dtype: float64
      player_name_fip  game_year          IP       FIP
2055  Sandy Alcantara       2022  228.666667  2.985609
2404       Logan Webb       2023  216.000000  3.162448
529      Zack Wheeler       2021  213.333333  2.588784
3080       Zac Gallen       2023  210.000000  3.264564
381       Gerrit Cole       2023  209.000000  3.164131
3207    Logan Gilbert       2024  208.666667  3.267119
1419   Walker Buehler       2021  207.666667  3.155588
2406       Logan Webb       2025  207.000000  2.603735
1251        Seth Lugo       2024  206.666667  3.248738
3     Adam Wainwright       2021  206.333333  3.664380


## Step 3. 저장

In [3]:
stats.to_csv("../data/processed/fip_2021_2025.csv", index=False)
print("저장 완료:", stats.shape)

저장 완료: (4366, 11)


## 결과 요약 (출력 삭제 전 기록)

- 시즌별 FIP 상수: 2021 3.170 / 2022 3.112 / 2023 3.255 / 2024 3.166 / 2025 3.135
  (FanGraphs 공개 상수와 거의 일치)
- 저장 행 수: 4,366 (투수-시즌)
- 이닝 상위 투수 FIP 예시: Sandy Alcantara 2022 (228.2이닝, 2.99), Gerrit Cole 2023 (209이닝, 3.16),
  Zack Wheeler 2021 (213.1이닝, 2.59)
- 이닝은 아웃 수 / 3으로 저장 (예: 228.67 = 228.2이닝)